# Textual Manifold-based Defense (TMD) on Colab Pro

One self-contained notebook for [Nguyen & Luu, EMNLP 2022](https://aclanthology.org/2022.emnlp-main.443/) (arXiv 2211.02878). It reimplements the pipeline from the repo in plain PyTorch (no wandb, no Lightning, no SLURM):

1. fine-tune BERT-base on a dataset (default AG News)
2. extract the BERT pooler embeddings `t = F(x)` and train the InfoGAN (architecture from the paper's Tables 7/8, hyperparameters from Table 6)
3. defend with on-manifold projection (sampling-based reconstruction, Eq. 8)
4. attack the vanilla and the TMD model with the paper's three word-substitution attacks (PWWS, TextFooler, BERT-Attack) **plus two character-level attacks (DeepWordBug, TextBugger)** that the paper never tests

**Read before trusting any number:**
- This notebook has not been run by its author. Expect to fix small API issues (TextAttack versions move).
- It uses the stock TextAttack recipes plus a max-words-perturbed constraint. The paper uses the TextDefender settings of Li et al. 2021 (K_max=50, similarity >= 0.84, query budget K*L), so the numbers will not match Table 1 exactly.
- The default `N_EVAL` is small so a first pass finishes. The paper uses 1000 examples per attack.
- Runtime: fine-tuning and InfoGAN are minutes on an A100. The attacks dominate (BERT-Attack and PWWS are the slowest, and TMD adds projection cost to every query).

Pick a GPU runtime first: Runtime > Change runtime type > A100 or L4.

In [ ]:
!pip -q install -U transformers datasets accelerate textattack tensorflow_hub pandas
# If pip reports a numpy / protobuf conflict, restart the runtime once (Runtime > Restart session) and re-run from here.

In [ ]:
import os, random, json, time, copy
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F

# ----------------------------- config -----------------------------
DATASET     = 'ag_news'            # 'ag_news' | 'imdb' | 'yelp_polarity'
MODEL_NAME  = 'bert-base-uncased'
SEED        = 42
FT_EPOCHS   = 3                    # paper fine-tunes for 10 epochs with an lr search; 3 is enough for a first pass
FT_LR       = 3e-5
FT_BS       = 32
GAN_EPOCHS  = 100                  # paper: 100
GAN_BS      = 1024
K_SAMPLES   = 20                   # sampling size k at inference (paper Fig. 3: best near 20)
N_EVAL      = 200                  # attacked examples per attack (paper: 1000)
ATTACKS     = ['pwws', 'textfooler', 'bertattack', 'deepwordbug', 'textbugger']
USE_DRIVE   = True                 # keep checkpoints on Google Drive so a disconnect does not lose the run

# Table 6 (BERT column) of the paper + max_len from Appendix A.1
CFG = {
    'ag_news':       dict(max_len=128, n_cls=4, K=50,  d=20, g_lr=1e-4, d_lr=1e-4, p_lr=0.0, dg=1, rho=0.3),
    'imdb':          dict(max_len=256, n_cls=2, K=50,  d=40, g_lr=1e-4, d_lr=1e-4, p_lr=0.0, dg=1, rho=0.1),
    'yelp_polarity': dict(max_len=256, n_cls=2, K=100, d=20, g_lr=1e-4, d_lr=9e-5, p_lr=0.0, dg=1, rho=0.1),
}[DATASET]

# Table 1 of the paper, BERT rows: (clean acc, PWWS, TextFooler, BERT-Attack)
PAPER = {
    'ag_news':       {'vanilla': (94.39, 39.2, 27.9, 39.0), 'TMD': (94.29, 70.0, 50.0, 55.2)},
    'imdb':          {'vanilla': (92.15, 6.5, 2.3, 1.0),    'TMD': (92.17, 38.7, 44.2, 33.7)},
    'yelp_polarity': {'vanilla': (95.28, 13.7, 10.5, 2.8),  'TMD': (95.24, 36.8, 40.9, 28.6)},
}[DATASET]

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), 'Select a GPU runtime first.'
dev = torch.device('cuda')
torch.backends.cudnn.benchmark = True
print(torch.cuda.get_device_name(0))

WORK = '/content/tmd_run'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/tmd_run'
WORK = os.path.join(WORK, DATASET)
os.makedirs(WORK, exist_ok=True)
print('artifacts in', WORK)

## 1. Data and fine-tuning

In [ ]:
from datasets import load_dataset
raw = load_dataset(DATASET)
if DATASET == 'yelp_polarity':   # paper: 63k-sample subset
    idx = random.Random(42).sample(range(len(raw['train'])), 63000)
    raw['train'] = raw['train'].select(idx)
split = raw['train'].train_test_split(0.1, seed=42)   # 10% validation, as in the paper
train_ds, val_ds, test_ds = split['train'], split['test'], raw['test']
print(len(train_ds), len(val_ds), len(test_ds))

from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
AMP = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

def batches(ds, bs, shuffle=False):
    idx = list(range(len(ds)))
    if shuffle: random.shuffle(idx)
    for i in range(0, len(idx), bs):
        b = ds[idx[i:i + bs]]
        yield b['text'], torch.tensor(b['label'])

def encode(texts):
    return tok(list(texts), truncation=True, max_length=CFG['max_len'], padding=True, return_tensors='pt').to(dev)

FT_DIR = os.path.join(WORK, 'finetuned')
if os.path.exists(os.path.join(FT_DIR, 'config.json')):
    print('loading fine-tuned model from', FT_DIR)
    model = AutoModelForSequenceClassification.from_pretrained(FT_DIR).to(dev)
else:
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=CFG['n_cls']).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=FT_LR)
    steps = FT_EPOCHS * ((len(train_ds) + FT_BS - 1) // FT_BS)
    sched = get_linear_schedule_with_warmup(opt, 0, steps)
    scaler = torch.cuda.amp.GradScaler(enabled=(AMP == torch.float16))

    @torch.no_grad()
    def accuracy(ds):
        model.eval(); ok = 0
        for x, y in batches(ds, 256):
            with torch.autocast('cuda', dtype=AMP):
                p = model(**encode(x)).logits.argmax(-1).cpu()
            ok += (p == y).sum().item()
        return ok / len(ds)

    best = 0.0
    for ep in range(FT_EPOCHS):
        model.train(); t0 = time.time()
        for x, y in batches(train_ds, FT_BS, shuffle=True):
            with torch.autocast('cuda', dtype=AMP):
                loss = model(**encode(x), labels=y.to(dev)).loss
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        va = accuracy(val_ds)
        print(f'epoch {ep}: val acc {va:.4f} ({time.time() - t0:.0f}s)')
        if va > best:
            best = va; model.save_pretrained(FT_DIR); tok.save_pretrained(FT_DIR)
    model = AutoModelForSequenceClassification.from_pretrained(FT_DIR).to(dev)
model.eval()
for p in model.parameters(): p.requires_grad_(False)

## 2. Text embeddings and InfoGAN

The embedding is BERT's pooler output (a tanh output in [-1, 1], which matches the generator's tanh). The classifier `C` is the model's own classification layer on that vector, so projecting the embedding and classifying needs no change to the language model.

In [ ]:
@torch.no_grad()
def pooled(texts):
    with torch.autocast('cuda', dtype=AMP):
        return model.bert(**encode(texts)).pooler_output.float()

def embed_ds(ds, name):
    path = os.path.join(WORK, f'emb_{name}.pt')
    if os.path.exists(path):
        return torch.load(path)
    E, Y = [], []
    for x, y in batches(ds, 256):
        E.append(pooled(x).cpu()); Y.append(y)
    out = (torch.cat(E), torch.cat(Y))
    torch.save(out, path)
    return out

E_tr, Y_tr = embed_ds(train_ds, 'train')
E_va, Y_va = embed_ds(val_ds, 'val')
E_te, Y_te = embed_ds(test_ds, 'test')
print(E_tr.shape, E_va.shape, E_te.shape)
with torch.no_grad():
    clean_acc_vanilla = (model.classifier(E_te.to(dev)).argmax(-1).cpu() == Y_te).float().mean().item()
print('vanilla clean acc on the full test set:', round(100 * clean_acc_vanilla, 2))

In [ ]:
# InfoGAN with learnable prior: architecture copied from src/models/generative_models/gan_based/infogan.py
class Gen(nn.Module):
    def __init__(self, latent_dim, code_dim):
        super().__init__()
        self.latent_dim, self.code_dim = latent_dim, code_dim
        self.net = nn.Sequential(
            nn.ConvTranspose1d(latent_dim + code_dim, 512, 32, 1, 0, bias=False), nn.BatchNorm1d(512), nn.ReLU(True),
            nn.ConvTranspose1d(512, 384, 4, 2, 1, bias=False), nn.BatchNorm1d(384), nn.ReLU(True),
            nn.ConvTranspose1d(384, 256, 4, 2, 1, bias=False), nn.BatchNorm1d(256), nn.ReLU(True),
            nn.ConvTranspose1d(256, 128, 4, 2, 1, bias=False), nn.BatchNorm1d(128), nn.ReLU(True),
            nn.ConvTranspose1d(128, 1, 5, 3, 1, bias=False), nn.Tanh())
    def forward(self, z, c):
        inp = torch.cat([z, F.one_hot(c, self.code_dim).float()], dim=-1).unsqueeze(-1)
        return self.net(inp).squeeze(1)

class Dis(nn.Module):
    def __init__(self, code_dim):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv1d(1, 128, 5, 3, 1, bias=False), nn.LeakyReLU(0.2, True),
            nn.Conv1d(128, 256, 4, 2, 1, bias=False), nn.BatchNorm1d(256), nn.LeakyReLU(0.2, True),
            nn.Conv1d(256, 384, 4, 2, 1, bias=False), nn.BatchNorm1d(384), nn.LeakyReLU(0.2, True),
            nn.Conv1d(384, 512, 4, 2, 1, bias=False), nn.BatchNorm1d(512), nn.LeakyReLU(0.2, True),
            nn.Conv1d(512, 768, 4, 2, 1, bias=False), nn.BatchNorm1d(768), nn.LeakyReLU(0.2, True))
        self.dis_head = nn.Conv1d(768, 1, 16, 1, 0, bias=False)
        self.enc_head = nn.Sequential(nn.Flatten(), nn.Linear(768 * 16, 128), nn.BatchNorm1d(128),
                                      nn.LeakyReLU(0.2, True), nn.Linear(128, code_dim))
    def forward(self, x):
        reps = self.body(x.unsqueeze(1))
        d_prob = torch.sigmoid(self.dis_head(reps).squeeze(-1))
        return d_prob, self.enc_head(reps)

class InfoGAN(nn.Module):
    def __init__(self, latent_dim, code_dim):
        super().__init__()
        self.latent_dim, self.code_dim = latent_dim, code_dim
        self.gen, self.dis = Gen(latent_dim, code_dim), Dis(code_dim)
        self.prior = nn.Parameter(torch.ones(code_dim) / code_dim)

@torch.no_grad()
def project(gan, t, k=K_SAMPLES, chunk=64):
    # Eq. 8: c_t = argmax Q(c|t); sample k z ~ N(0, I); keep the generated embedding closest to t (L2).
    gan.eval()
    outs, losses = [], []
    for i in range(0, t.size(0), chunk):
        x = t[i:i + chunk]; bs = x.size(0)
        c = gan.dis(x)[1].argmax(-1)
        z = torch.randn(bs * k, gan.latent_dim, device=x.device)
        fake = gan.gen(z, c[:, None].expand(bs, k).reshape(-1)).view(bs, k, -1)
        d = torch.norm(fake - x[:, None], dim=-1)
        m = d.argmin(1)
        outs.append(fake[torch.arange(bs), m]); losses.append(d.min(1).values)
    return torch.cat(outs), torch.cat(losses)

def train_gan():
    gan = InfoGAN(CFG['d'], CFG['K']).to(dev)
    og = torch.optim.Adam(gan.gen.parameters(), lr=CFG['g_lr'], betas=(0.5, 0.999))
    od = torch.optim.Adam(gan.dis.parameters(), lr=CFG['d_lr'], betas=(0.5, 0.999))
    op = torch.optim.Adam([gan.prior], lr=CFG['p_lr'], betas=(0.5, 0.999)) if CFG['p_lr'] > 0 else None
    Etr, Eva = E_tr.to(dev), E_va[:5000].to(dev)
    best, best_state, pl_counter = 1e9, None, 1.0
    for ep in range(GAN_EPOCHS):
        gan.train(); perm = torch.randperm(len(Etr), device=dev)
        for bi, i in enumerate(range(0, len(perm), GAN_BS)):
            real = Etr[perm[i:i + GAN_BS]]; n = real.size(0)
            if n < 2: continue
            z = torch.randn(n, gan.latent_dim, device=dev)
            c = torch.multinomial(torch.softmax(gan.prior, -1).detach(), n, replacement=True)
            one, zero = torch.ones(n, 1, device=dev), torch.zeros(n, 1, device=dev)
            fake = gan.gen(z, c).detach()
            pr, _ = gan.dis(real); pf, ef = gan.dis(fake)
            d_loss = (F.binary_cross_entropy(pr, one) + F.binary_cross_entropy(pf, zero)) / 2 + F.cross_entropy(ef, c)
            od.zero_grad(); d_loss.backward(); od.step()
            if bi % CFG['dg'] == 0:
                pf, ef = gan.dis(gan.gen(z, c))
                g_loss = F.binary_cross_entropy(pf, one) + F.cross_entropy(ef, c)
                og.zero_grad(); g_loss.backward(); og.step()
            if op is not None:
                with torch.no_grad(): e_real = gan.dis(real)[1]
                lp = F.log_softmax(gan.prior, -1)
                ce = -(torch.softmax(e_real, -1) * lp).sum(-1).mean()
                ent = -(lp.exp() * lp).sum()
                pl_counter *= 0.999
                p_loss = ce - pl_counter * 100.0 * ent
                op.zero_grad(); p_loss.backward(); op.step()
        if ep % 5 == 4 or ep == GAN_EPOCHS - 1:
            rec = project(gan, Eva)[1].mean().item()
            print(f'gan epoch {ep + 1}: val reconstruction loss {rec:.4f}')
            if rec < best:
                best, best_state = rec, copy.deepcopy(gan.state_dict())
    gan.load_state_dict(best_state)
    return gan.eval()

GAN_PATH = os.path.join(WORK, 'infogan.pt')
gan = InfoGAN(CFG['d'], CFG['K']).to(dev)
if os.path.exists(GAN_PATH):
    gan.load_state_dict(torch.load(GAN_PATH)); gan.eval(); print('loaded InfoGAN')
else:
    gan = train_gan(); torch.save(gan.state_dict(), GAN_PATH)

In [ ]:
# Sanity check on the full test set: clean accuracy with the projection, and how far clean embeddings sit from the manifold.
with torch.no_grad():
    Et = E_te.to(dev)
    t_hat, rec = project(gan, Et)
    clean_acc_tmd = (model.classifier(t_hat).argmax(-1).cpu() == Y_te).float().mean().item()
print('TMD clean acc:', round(100 * clean_acc_tmd, 2), '| vanilla:', round(100 * clean_acc_vanilla, 2))
print('mean d_G(t) on clean test embeddings:', round(rec.mean().item(), 3))
print('paper clean acc (BERT):', PAPER['vanilla'][0], 'vanilla /', PAPER['TMD'][0], 'TMD')

## 3. Attacks (including the character-level ones the paper skips)

The same random test examples are attacked on both the vanilla model and the TMD model. Accuracy under attack counts originally misclassified examples as wrong, as TextAttack does.

In [ ]:
import textattack
from textattack import Attacker, AttackArgs
from textattack.datasets import Dataset
from textattack.models.wrappers import ModelWrapper
from textattack.attack_recipes import (PWWSRen2019, TextFoolerJin2019, BERTAttackLi2020,
                                       DeepWordBugGao2018, TextBuggerLi2018)
from textattack.constraints.overlap import MaxWordsPerturbed

RECIPES = {'pwws': PWWSRen2019, 'textfooler': TextFoolerJin2019, 'bertattack': BERTAttackLi2020,
           'deepwordbug': DeepWordBugGao2018, 'textbugger': TextBuggerLi2018}

class Victim(ModelWrapper):
    def __init__(self, use_tmd):
        self.model, self.use_tmd = model, use_tmd
    @torch.no_grad()
    def __call__(self, text_input_list):
        texts, out = list(text_input_list), []
        for i in range(0, len(texts), 64):
            t = pooled(texts[i:i + 64])
            if self.use_tmd:
                t = project(gan, t)[0]
            out.append(model.classifier(t).cpu())
        return torch.cat(out)

rng = random.Random(0)
sel = rng.sample(range(len(test_ds)), N_EVAL)
pairs = [(test_ds[i]['text'], int(test_ds[i]['label'])) for i in sel]

def run_attack(name, use_tmd):
    attack = RECIPES[name].build(Victim(use_tmd))
    try:
        attack.constraints.append(MaxWordsPerturbed(max_percent=CFG['rho']))
    except Exception as e:
        print('  could not add MaxWordsPerturbed:', e)
    args = AttackArgs(num_examples=len(pairs), random_seed=SEED, shuffle=False, silent=True, disable_stdout=True)
    t0 = time.time()
    res = Attacker(attack, Dataset(pairs), args).attack_dataset()
    kinds = [type(r).__name__ for r in res]
    n = len(res); failed = kinds.count('FailedAttackResult')
    skipped = kinds.count('SkippedAttackResult'); succ = kinds.count('SuccessfulAttackResult')
    return dict(acc_under_attack=100 * failed / n, success_rate=100 * succ / max(1, n - skipped),
                orig_acc=100 * (n - skipped) / n, avg_queries=float(np.mean([r.num_queries for r in res])),
                minutes=(time.time() - t0) / 60)

RES_PATH = os.path.join(WORK, 'attack_results.json')
results = json.load(open(RES_PATH)) if os.path.exists(RES_PATH) else {}
for name in ATTACKS:
    for tag, use_tmd in (('vanilla', False), ('TMD', True)):
        key = f'{tag}/{name}'
        if key in results: continue
        print('running', key, flush=True)
        try:
            results[key] = run_attack(name, use_tmd)
            print('  ', {k: round(v, 2) for k, v in results[key].items()}, flush=True)
        except Exception as e:
            print('  FAILED:', repr(e)[:300]); continue
        json.dump(results, open(RES_PATH, 'w'), indent=1)

## 4. Results

In [ ]:
rows = []
for name in ATTACKS:
    v, t = results.get(f'vanilla/{name}'), results.get(f'TMD/{name}')
    if v and t:
        rows.append(dict(attack=name, vanilla_acc=round(v['acc_under_attack'], 1), TMD_acc=round(t['acc_under_attack'], 1),
                         gain=round(t['acc_under_attack'] - v['acc_under_attack'], 1),
                         TMD_success_rate=round(t['success_rate'], 1), TMD_avg_queries=round(t['avg_queries'])))
df = pd.DataFrame(rows)
display(df)
print(f'N_EVAL={N_EVAL}; with this many examples a difference of a few points is within noise (~+-3.5 points at N=200).')
print(f'clean acc (full test set): vanilla {100 * clean_acc_vanilla:.2f}, TMD {100 * clean_acc_tmd:.2f}')

ref = pd.DataFrame({'': ['clean', 'PWWS', 'TextFooler', 'BERT-Attack'],
                    'paper vanilla': PAPER['vanilla'], 'paper TMD': PAPER['TMD']})
print('Paper Table 1 (BERT, ' + DATASET + '), different attack settings, 1000 examples:')
display(ref)
df.to_csv(os.path.join(WORK, 'summary.csv'), index=False)

## How to read this

- **PWWS / TextFooler / BERT-Attack** rows are comparable in kind (not exactly in setting) to the paper's Table 1.
- **DeepWordBug / TextBugger** rows are the extension: character-level edits create typos, which change tokenization and the BERT embedding in a way the manifold was never trained on. TMD could look better or worse than on word swaps, and the paper gives no evidence either way.
- To change dataset or model, edit the config cell and run all. Checkpoints and results are cached per dataset in `WORK`, so a disconnected session resumes where it stopped.
- An adaptive attacker who knows about the projection (for example by attacking through the generator, or averaging over the random z) is a separate and stronger test that this notebook does not run.